![Python](https://img.shields.io/badge/python-3.9-blue)
![Status: Pending Migration](https://img.shields.io/badge/status-pending%20migration-orange)

<a id="table-of-contents"></a>
# 🧾 Topic Modeling
- [📄 Introduction](#introduction)  
- [📦 Text Preprocessing for Topic Modeling](#text-preprocessing)  
- [📊 Latent Dirichlet Allocation (LDA)](#lda)  
- [🧮 Non-negative Matrix Factorization (NMF)](#nmf)  
- [🧠 Interpreting Topics](#interpret-topics)  
- [📈 Topic Coherence & Quality Metrics](#topic-coherence)  
- [🗂️ Visualizing Topics](#visualization)  
- [🧪 Edge Cases & Troubleshooting](#edge-cases)
___

<a id="introduction"></a>  
# 📄 Introduction  

<details><summary><strong>📖 Click to Expand</strong></summary>

##### 🧠 What You'll Learn in This Notebook

In this notebook, we will explore **Topic Modeling**, a powerful set of techniques in Natural Language Processing (NLP) used to automatically uncover the hidden thematic structure in a collection of documents.

Topic modeling is considered **unsupervised learning**, as it attempts to detect latent patterns in textual data without labeled outputs.

We'll specifically cover two popular algorithms:

- 🔵 **Latent Dirichlet Allocation (LDA)** – a probabilistic generative model that assumes each document is a mixture of topics, and each topic is a distribution over words.
- 🟣 **Non-negative Matrix Factorization (NMF)** – a linear algebra-based method that decomposes a document-term matrix into interpretable topic and word matrices.

By the end of this notebook, you’ll be able to:
- Preprocess and vectorize a corpus for topic modeling.
- Apply both LDA and NMF to extract topics.
- Interpret topics using top words and document-topic distributions.
- Compare LDA and NMF across interpretability and use cases.

We'll demonstrate everything on a real-world dataset (you can choose `amazon.csv` for product reviews or another corpus if preferred).

</details>


[Back to the top](#table-of-contents)
___


<a id="text-preprocessing"></a>  
# 📦 Text Preprocessing for Topic Modeling

#### 🧼 Cleaning for Unsupervised Models  
<a id="cleaning-unsupervised"></a>  



<details><summary><strong>📖 Click to Expand</strong></summary>

##### 🧹 Why Cleaning Matters More for Topic Modeling

In supervised learning (e.g., classification), models can often learn to ignore noise if the signal is strong. But in **unsupervised modeling like LDA or NMF**, there’s no target to guide learning — **so noisy, rare, or overly common terms can distort topic discovery.**

Here’s what we’ll do to prep the data:
- **Lowercase** the text
- **Remove punctuation & digits**
- **Drop stopwords** (e.g., “the”, “is”, “and”)
- **Apply lemmatization** (more consistent than stemming for topic modeling)

We aim to preserve **semantically meaningful tokens** that help identify thematic clusters.  

</details>


In [4]:
import pandas as pd
import re
import nltk

# Download only what we still use
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Load data
df = pd.read_csv('datasets/amazon.csv')
df = df.dropna(subset=['text']).reset_index(drop=True)

# Setup stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# 🔧 Updated cleaning function (no NLTK tokenizer)
def clean_text(text):
    try:
        text = text.lower()
        text = re.sub(r'[^a-z\s]', '', text)                      # Remove non-letters
        tokens = re.findall(r'\b[a-z]{3,}\b', text)               # Only words with 3+ letters
        tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
        return ' '.join(tokens)
    except Exception as e:
        print(f"[Error cleaning text]: {text[:50]}...\nReason: {e}")
        return ""

# Apply cleaning
df['clean_text'] = df['text'].apply(clean_text)

# Preview
df[['text', 'clean_text']].head()


ModuleNotFoundError: No module named 'pandas'

#### 🔁 Vectorization with Count/TF-IDF  
<a id="vectorization-topic"></a>  

<details><summary><strong>📖 Click to Expand</strong></summary>

##### 🧾 Choosing Between Count and TF-IDF

Topic models don’t work on raw text — they require a **document-term matrix** (rows: docs, columns: words).  
Two common vectorization strategies are:

- **Count Vectorizer**: Keeps track of raw word counts
- **TF-IDF Vectorizer**: Weights down common words and boosts rare, distinctive words

> 🧠 For **LDA**, CountVectorizer is often preferred because it aligns with the probabilistic model's assumptions.
>  
> 🧠 For **NMF**, TF-IDF is usually better — it stabilizes learning by de-emphasizing high-frequency noise.

We’ll create **both** and store them for later modeling.

</details>



In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Use only clean text
texts = df['clean_text'].tolist()

# Count Vectorizer
count_vectorizer = CountVectorizer(max_df=0.95, min_df=2, max_features=1000)
dtm_count = count_vectorizer.fit_transform(texts)

# TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, max_features=1000)
dtm_tfidf = tfidf_vectorizer.fit_transform(texts)

# Show dimensions
print("Count DTM shape:", dtm_count.shape)
print("TF-IDF DTM shape:", dtm_tfidf.shape)


[Back to the top](#table-of-contents)
___


<a id="lda"></a>  
# 📊 Latent Dirichlet Allocation (LDA)



#### 🧠 LDA Intuition & Assumptions  
<a id="lda-intuition"></a>  

<details><summary><strong>📖 Click to Expand</strong></summary>

##### 🤯 What is LDA?

**Latent Dirichlet Allocation (LDA)** is a probabilistic generative model used for topic modeling.

It assumes:
- Every **document is a mixture of topics**
- Every **topic is a mixture of words**
- The process is governed by **Dirichlet distributions**, which enforce sparsity (i.e., each doc has only a few topics, each topic uses a few words heavily)

> Imagine a set of news articles:  
> LDA might infer that one article is 80% “sports” and 20% “politics,” and that the “sports” topic is mostly composed of words like *game*, *team*, *score*.

##### 🧰 LDA Is Useful When:
- You want to **summarize large corpora**
- You want to **group documents** without labels
- You care about **interpretable topics** (vs. black-box clusters)

</details>




#### ⚙️ LDA Implementation (Sklearn / Gensim)  
<a id="lda-implementation"></a>  



<details><summary><strong>📖 Click to Expand</strong></summary>

##### 🧪 LDA via Scikit-Learn vs. Gensim

There are two major ways to implement LDA:

- 🤖 **Scikit-Learn**: Fast, matrix-based, works well with pipelines and TF/Count matrices  
- 📚 **Gensim**: More customizable, handles large corpora with streaming, shows better topic diagnostics

We’ll start with **Scikit-Learn** using the CountVectorizer DTM (created earlier).

</details>


In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

# We'll reuse earlier DTM (if not run, re-vectorize here)
from sklearn.feature_extraction.text import CountVectorizer

texts = df['clean_text'].tolist()

count_vectorizer = CountVectorizer(max_df=0.95, min_df=2, max_features=1000)
dtm_count = count_vectorizer.fit_transform(texts)

# Fit LDA
lda_model = LatentDirichletAllocation(n_components=10, random_state=42)
lda_model.fit(dtm_count)

# Display top words per topic
def display_topics(model, feature_names, top_n=10):
    for topic_idx, topic in enumerate(model.components_):
        top_features = [feature_names[i] for i in topic.argsort()[:-top_n - 1:-1]]
        print(f"Topic {topic_idx + 1}: {', '.join(top_features)}")

print("🧠 Top Words per Topic:\n")
display_topics(lda_model, count_vectorizer.get_feature_names_out())


#### 🔍 Tuning `n_topics`  
<a id="lda-tuning"></a>  

<details><summary><strong>📖 Click to Expand</strong></summary>

##### 🎛️ How Many Topics to Use?

Choosing the number of topics (`n_components`) is more art than science. Too few → broad, vague topics. Too many → redundant, noisy ones.

You can tune this by:

- 📉 **Perplexity** (lower is better)  
- 📈 **Coherence score** (via Gensim, higher is better)  
- 👁️‍🗨️ **Manual interpretability** (are the topics actually meaningful?)

For now, we’ll loop through a few values of `k` and inspect **perplexity scores**.

</details>


In [ ]:
# import matplotlib.pyplot as plt

# scores = []
# k_values = list(range(2, 21, 2))

# for k in k_values:
#     lda = LatentDirichletAllocation(n_components=k, random_state=42)
#     lda.fit(dtm_count)
#     perplexity = lda.perplexity(dtm_count)
#     scores.append(perplexity)

# # Plot
# plt.plot(k_values, scores, marker='o')
# plt.title("Perplexity vs Number of Topics")
# plt.xlabel("n_topics")
# plt.ylabel("Perplexity (lower is better)")
# plt.grid(True)
# plt.show()


[Back to the top](#table-of-contents)
___


<a id="nmf"></a>  
# 🧮 Non-negative Matrix Factorization (NMF)



#### 📘 NMF vs LDA  
<a id="nmf-comparison"></a>  

<details><summary><strong>📖 Click to Expand</strong></summary>

##### 🔍 How NMF Differs from LDA

While LDA is a **generative probabilistic model**, **NMF** is a **linear algebra decomposition**.  
NMF factorizes the **document-term matrix (V)** into two lower-rank matrices:

$$
V \approx W \times H
$$


- **W**: Document-topic matrix  
- **H**: Topic-word matrix  
- All entries are non-negative (no subtraction of concepts)

##### 🔄 Summary: NMF vs LDA

| Feature | LDA | NMF |
|--------|-----|-----|
| Type | Probabilistic | Matrix factorization |
| Input | Count | TF-IDF |
| Output | Topic probabilities | Topic weights |
| Speed | Slower | Faster |
| Interpretability | Often better for LDA | Clean when using TF-IDF |
| Library | sklearn / gensim | sklearn |

</details>




#### ⚙️ NMF Implementation  
<a id="nmf-implementation"></a>  

<details><summary><strong>📖 Click to Expand</strong></summary>

##### 🛠️ NMF in Scikit-Learn

We’ll use `TfidfVectorizer` (made earlier) to build a TF-IDF document-term matrix and then apply `NMF` from `sklearn.decomposition`.

NMF tends to yield **sharper, cleaner topic-word associations** than LDA, especially on product review or short-form data.

</details>


In [ ]:
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorize with TF-IDF (if not already)
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, max_features=1000)
dtm_tfidf = tfidf_vectorizer.fit_transform(df['clean_text'])

# Fit NMF
nmf_model = NMF(n_components=10, random_state=42)
nmf_model.fit(dtm_tfidf)

# Display top words
def display_nmf_topics(model, feature_names, top_n=10):
    for idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-top_n - 1:-1]]
        print(f"Topic {idx + 1}: {', '.join(top_words)}")

print("📌 NMF Top Words per Topic:\n")
display_nmf_topics(nmf_model, tfidf_vectorizer.get_feature_names_out())


[Back to the top](#table-of-contents)
___


<a id="interpret-topics"></a>  
# 🧠 Interpreting Topics



#### 📝 Top Words per Topic  
<a id="top-words-topic"></a>  

<details><summary><strong>📖 Click to Expand</strong></summary>

##### 🧾 What Makes a Topic Interpretable?

Once we extract topic distributions, we interpret each topic by inspecting its **top contributing words**.

A topic with high-weight words like:

**Topic 2 →** `delivery`, `late`, `order`, `package`, `customer`

likely refers to **logistics or shipping complaints**, even though the model doesn’t “know” that.

🧠 Tip: Focus on the **top 5–10 words** per topic. If they cluster around a shared theme, the topic is usable.

</details>




In [ ]:
print("🔁 Top Words from LDA Topics:\n")
display_topics(lda_model, count_vectorizer.get_feature_names_out())

print("\n🧮 Top Words from NMF Topics:\n")
display_nmf_topics(nmf_model, tfidf_vectorizer.get_feature_names_out())


#### 🧵 Assigning Topics to Documents  
<a id="assign-topics"></a>  

<details><summary><strong>📖 Click to Expand</strong></summary>

##### 📌 Tagging Each Document with a Dominant Topic

After training, we can inspect the **document-topic distribution matrix** to assign a dominant topic to each doc.

- In **LDA**, this is a **probability distribution** (sums to 1)  
- In **NMF**, it’s **weighted scores** (non-negative, sparse)

We assign the **argmax topic** — i.e., whichever topic is strongest for that document.

These assignments can be used for:
- Clustering
- Filtering
- Tagging
- Downstream supervised tasks

</details>



In [ ]:
# LDA assignments
lda_doc_topic_dist = lda_model.transform(dtm_count)
df['lda_topic'] = lda_doc_topic_dist.argmax(axis=1) + 1  # +1 for human-friendly topic numbers

# NMF assignments
nmf_doc_topic_dist = nmf_model.transform(dtm_tfidf)
df['nmf_topic'] = nmf_doc_topic_dist.argmax(axis=1) + 1

# Preview tagged docs
df[['text', 'lda_topic', 'nmf_topic']].head(10)


[Back to the top](#table-of-contents)
___


<a id="topic-coherence"></a>  
# 📈 Topic Coherence & Quality Metrics



#### 📊 Coherence Scores  
<a id="coherence-scores"></a>  

<details><summary><strong>📖 Click to Expand</strong></summary>

##### 📐 What is Topic Coherence?

Topic Coherence measures **how semantically related** the top words in a topic are. Unlike perplexity (which is purely probabilistic), coherence correlates better with **human judgment** of topic quality.

A coherent topic has top words that **tend to co-occur** and form a clear theme.

💬 Example:

- Coherent: `battery`, `charger`, `voltage`, `cable`, `adapter`
- Incoherent: `battery`, `movie`, `shirt`, `charger`, `weather`

We’ll use **Gensim’s `CoherenceModel`** with the `'c_v'` metric, which works well on short texts and supports tokenized input.

</details>


In [ ]:
# !pip3 install --no-cache-dir gensim==4.3.2
# !pip3 install --upgrade --force-reinstall scipy

!pip3 uninstall scipy numpy gensim --yes
!pip3 install numpy==1.24.4 scipy==1.10.1 gensim==4.3.2




In [ ]:
import sys
print(sys.executable)


In [ ]:
!pip3 install scipy

In [ ]:
# from gensim.models import CoherenceModel
# from gensim.corpora import Dictionary


# !pip3 uninstall gensim scipy numpy --yes
# !pip3 install numpy==1.24.4 scipy==1.10.1 gensim==4.3.2

# from scipy.linalg import triu
# from gensim.models import CoherenceModel
# from gensim.corpora import Dictionary



from scipy.linalg import triu

print(triu([[1, 2], [3, 4]]))


In [ ]:
import sys
print(sys.executable)


In [ ]:
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary

# Tokenize cleaned text
tokenized_docs = [doc.split() for doc in df['clean_text']]

# Gensim Dictionary and Corpus
id2word = Dictionary(tokenized_docs)
corpus = [id2word.doc2bow(text) for text in tokenized_docs]

# Convert LDA model from sklearn to Gensim format for coherence
lda_topics = []
for topic_weights in lda_model.components_:
    top_word_ids = topic_weights.argsort()[:-11:-1]
    lda_topics.append([count_vectorizer.get_feature_names_out()[i] for i in top_word_ids])

# Compute coherence
coherence_lda = CoherenceModel(
    topics=lda_topics,
    texts=tokenized_docs,
    dictionary=id2word,
    coherence='c_v'
)

coherence_score = coherence_lda.get_coherence()
print(f"🧠 LDA Coherence Score (c_v): {coherence_score:.4f}")


#### 🎯 Perplexity & Limitations  
<a id="perplexity"></a>  

<details><summary><strong>📖 Click to Expand</strong></summary>

##### 🎭 Why Perplexity Alone Isn’t Enough

**Perplexity** measures how well a model predicts a sample. Lower values suggest the model is better at “compressing” or explaining the data.

But here’s the catch:

> “A model with low perplexity may generate incoherent topics.”

🧨 Why? Because perplexity focuses on **word-level prediction**, not **semantic quality**.

That’s why topic modeling evaluation is tricky — it needs a balance of:

- **Perplexity** → Statistical fit  
- **Coherence** → Interpretability  
- **Manual inspection** → Real-world value

🧠 Use coherence for deciding `n_topics`, and perplexity as a sanity check — **never alone**.

</details>


[Back to the top](#table-of-contents)
___


<a id="visualization"></a>  
# 🗂️ Visualizing Topics



#### 📦 pyLDAvis  
<a id="pyldavis"></a>  



#### 🖼️ Wordclouds by Topic  
<a id="wordclouds"></a>  


[Back to the top](#table-of-contents)
___


<a id="edge-cases"></a>  
# 🧪 Edge Cases & Troubleshooting



#### ⚠️ Short Texts & Low Topic Separation  
<a id="short-texts"></a>  



#### 🧪 Poor Topic Coherence  
<a id="poor-coherence"></a>  


[Back to the top](#table-of-contents)
___
